# ESMBiologist Demo

Zero-shot peptide scoring with ESM-2 embeddings.

In [1]:
import sys, os

project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from peptide_pipeline.biologist.esm_biologist_zscore import ESMBiologistZscore
import transformers

/home/arthur/Documents/M2/pfe/multi-expert-peptide-pipeline/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Define peptides

In [2]:
REFERENCE_PEPTIDE = "FKIGGFIKKLWRSLLA"

candidate_peptides = [
    # -----------------------------------------------------------------------
    # CATEGORY 1: Brand New Peptides with Theoretical Activity against S. mutans
    # (Designed to be cationic and amphipathic to disrupt bacterial cell walls)
    # -----------------------------------------------------------------------
    "RWRRWVRRWW",      # Brand new - S. mutans target (Tryptophan/Arginine rich)
    "KLLKKVWKKLLK",    # Brand new - S. mutans target (Alpha-helical propensity)
    "FWKKLIRRKW",      # Brand new - S. mutans target (Hydrophobic core)
    "KWKLAFKKIKK",     # Brand new - S. mutans target (High Lysine content)
    "RRWWILLKWW",      # Brand new - S. mutans target (Strong lipid-anchoring)
    "FLRKVLRKFLRK",    # Brand new - S. mutans target (Amphipathic repeat)
    "VRLIRKWLKKVV",    # Brand new - S. mutans target (Membrane inserting)
    "WWKKALRRWW",      # Brand new - S. mutans target (Cationic flanking)
    "ILKKWVKKLKLL",    # Brand new - S. mutans target (Leucine zipper-like)
    "RWRARARW",        # Brand new - S. mutans target (Short, rigid sequence)
    "KKLLFWRRVL",      # Brand new - S. mutans target (Mixed hydrophobic/cationic)
    "WLRRIWSRLW",      # Brand new - S. mutans target (Tryptophan-anchored)
    "KWKWWVKLLK",      # Brand new - S. mutans target (Broad amphipathicity)
    "FFRRLKHLFF",      # Brand new - S. mutans target (Phenylalanine caps)
    "KLLWRQKWL",       # Brand new - S. mutans target (Moderate flexibility)

    # -----------------------------------------------------------------------
    # CATEGORY 2: Known Peptides with Activity against S. mutans
    # (Includes specific STAMPs and broad-spectrum oral AMPs)
    # -----------------------------------------------------------------------
    "RLLKRFKHLFK", # B-36
    "KLLQTFKQIFR", # B-39
    "KLFKRWKHLFR", # B-35
    "TFFRLFNRSFTQALGKGGGKNLRIIRKGIHIIKKY", # Known - C16G2 (Highly specific STAMP targeting S. mutans)
    "GLLWHLLHHLLH",                          # Known - GH12 (De novo AMP against oral streptococci)
    "AKRHHGYKRKFH",                          # Known - PAC-113 (Histatin-5 derived oral AMP)
    "FIKDFIERF",                             # Known - Sm8 (Specific S. mutans binding/targeting domain)
    "WWYNWWQDW",                             # Known - Sm4 (Specific S. mutans binding/targeting domain)
    "KNLRRIIRKGIHIIKKY",                     # Known - G2 (Killing domain portion of the C16G2 STAMP)
    "TFFRLFNRSFTQALGK",                      # Known - CSPC16 (S. mutans competence-stimulating peptide fragment)
    "LLGDFFRKSKEKIGKEFKRIVQRIKDFLRNLVPRTES", # Known - LL-37 (Human cathelicidin, broad oral bacterial activity)
    "GIGAVLKVLTTGLPALISWIKRKRQQ",            # Known - Melittin (Bee venom, highly active membrane disruptor)
    "GIGKFLHSAKKFGKAFVGEIMNS",               # Known - Magainin 2 (Frog skin AMP, broad-spectrum disruptor)

    # -----------------------------------------------------------------------
    # CATEGORY 3: Brand New Peptides with Random Activity
    # (Designed intentionally without standard AMP structural constraints)
    # -----------------------------------------------------------------------
    "DQEEDTGEYDS",     # Brand new - Random (Highly acidic, repels negative membranes)
    "PPHLMPMQNQ",      # Brand new - Random (Proline/Methionine rich, likely unstructured)
    "SSTTNYGQQQ",      # Brand new - Random (Highly polar, neutral charge)
    "CVDDMCYEPE",      # Brand new - Random (Disulfide potential, negatively charged)
    "MVNPTQSSLI",      # Brand new - Random (Lacks distinct secondary structure)
    "YYHEEDDGHH",      # Brand new - Random (Theoretical metal-binding potential)
    "AQVTPAQVTG",      # Brand new - Random (High turn/bend propensity)
    "LSEDENFDYE",      # Brand new - Random (Heavily anionic, no binding affinity to bacteria)
    "MMQPQPPPQ",       # Brand new - Random (Poly-proline/glutamine tract)
    "TSNGSVTVTQ",      # Brand new - Random (Beta-sheet potential, uncharged)
    "ECECECECEE",      # Brand new - Random (Artificial, repeating negative polymer)
    "NNQQNQQNQN",      # Brand new - Random (Prion-like polar aggregation sequence)
    "AAGGDDAAGG",      # Brand new - Random (Highly flexible and acidic)
    "HPHPHPHPHP",      # Brand new - Random (Alternating histidine/proline)
    "YEEVDDGEEE",      # Brand new - Random (Purely electrostatic repulsor for Gram-positives)

    # -----------------------------------------------------------------------
    # CATEGORY 4: Known Peptides with Random Activity
    # (Documented biological peptides with non-antimicrobial functions)
    # -----------------------------------------------------------------------
    "RPPGFSPFR",                     # Known - Bradykinin (Vasodilator, lowers blood pressure)
    "CYIQNCPLG",                     # Known - Oxytocin (Hormone, social bonding/reproduction)
    "DRVYIHPF",                      # Known - Angiotensin II (Vasoconstrictor, raises blood pressure)
    "RPKPQQFFGLM",                   # Known - Substance P (Neuropeptide, pain perception)
    "YGGFL",                         # Known - Leu-Enkephalin (Endogenous opioid/painkilller)
    "AGCKNFFWKTFTSC",                # Known - Somatostatin (Hormone, inhibits endocrine release)
    "HSQGTFTSDYSKYLDSRRAQDFVQWLMNT", # Known - Glucagon (Hormone, raises blood sugar)
    "GIVEQCCTSICSLYQLENYCN",         # Known - Insulin A-chain (Hormone fragment, metabolism)
    "CYFQNCPRG",                     # Known - Vasopressin (Antidiuretic hormone, water retention)
    "HSDAVFTDNYTRLRKQMAVKKYLNSILN"   # Known - VIP (Vasoactive Intestinal Peptide, smooth muscle relaxant)
]

print(f"Reference : {REFERENCE_PEPTIDE}")
print(f"Candidates: {len(candidate_peptides)} peptides")
for i in range(min(len(candidate_peptides)-1, 10)):
    print(f"  {candidate_peptides[i]}")

Reference : FKIGGFIKKLWRSLLA
Candidates: 53 peptides
  RWRRWVRRWW
  KLLKKVWKKLLK
  FWKKLIRRKW
  KWKLAFKKIKK
  RRWWILLKWW
  FLRKVLRKFLRK
  VRLIRKWLKKVV
  WWKKALRRWW
  ILKKWVKKLKLL
  RWRARARW


## Instantiate ESMBiologist

In [ ]:
biologist = ESMBiologistZscore(
    reference_peptide=REFERENCE_PEPTIDE,
    model_name="facebook/esm2_t33_650M_UR50D",
)
print(biologist)

NameError: name 'ESMBiologist2' is not defined

## score_peptides — cosine similarity to the reference

Scores are in **[0, 1]**: 1.0 = identical embedding, 0.5 = orthogonal, 0.0 = antipodal.

In [ ]:
# Score all candidates
scores = biologist.score_peptides(candidate_peptides)

print(f"\n{'Peptide':<30} {'Score':>7}")
print("-" * 40)
for pep, sc in sorted(zip(candidate_peptides, scores), key=lambda x: -x[1]):
    print(f"{pep:<30} {sc:.4f}")


Peptide                          Score
----------------------------------------
RLLKRFKHLFK                    0.8483
KLLQTFKQIFR                    0.8276
KLLKKVWKKLLK                   0.8150
FLRKVLRKFLRK                   0.7913
KNLRRIIRKGIHIIKKY              0.7796
FIKDFIERF                      0.7759
KLFKRWKHLFR                    0.7666
VRLIRKWLKKVV                   0.7535
LLGDFFRKSKEKIGKEFKRIVQRIKDFLRNLVPRTES 0.7502
GIGKFLHSAKKFGKAFVGEIMNS        0.7175
GIGAVLKVLTTGLPALISWIKRKRQQ     0.7098
FFRRLKHLFF                     0.7006
TFFRLFNRSFTQALGK               0.6655
FWKKLIRRKW                     0.6623
CYFQNCPRG                      0.6384
KKLLFWRRVL                     0.6331
RPKPQQFFGLM                    0.6165
AKRHHGYKRKFH                   0.6079
KWKLAFKKIKK                    0.5975
ILKKWVKKLKLL                   0.5971
YGGFL                          0.5781
DRVYIHPF                       0.5750
RPPGFSPFR                      0.5519
HSDAVFTDNYTRLRKQMAVKKYLNSILN   0.5412


## predict_activity with an alternative reference

In [ ]:
TEMP_REFERENCE = "KLFKRWKHLFR"

activity_scores = biologist.predict_activity(candidate_peptides, context=TEMP_REFERENCE)

print(f"{'Peptide':<30} {'Activity':>7}")
print("-" * 40)
for pep, sc in sorted(zip(candidate_peptides, activity_scores), key=lambda x: -x[1]):
    print(f"{pep:<30} {sc:.4f}")

NameError: name 'biologist' is not defined